# Stage B validation — RF vs kriging vs regression-kriging head-to-head

Side-by-side comparison of the three gap-fill candidates against AERONET:
- **`rf`** — B2 covariate-only Random Forest.
- **`kriging`** — B1 ST kriging of the (AOD − CAMS) residual.
- **`rf_rk`** — B3 regression kriging: RF drift + kriged RF residual. Should
  inherit kriging's accuracy near observations *and* the RF baseline in deep
  gaps (kriged correction → 0 ⇒ falls back to ŷ_rf).

Two flavours of comparison:
1. **Side-by-side panels** — independent metric panels for each candidate.
2. **Paired skill** — restricts to *identical* `(slot, site)` keys present in both candidates, then runs a paired t-test on `|error|`.  Avoids the trap where one candidate is graded on easier days than the other.

In [ ]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))
import validate as vb
import config as cfg

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

## Extract blind pairs for both candidates

In [ ]:
pairs_rf = vb.aeronet_pairs(START, END, candidate='rf',      blind_only=True)
pairs_kr = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=True)
pairs_rk = vb.aeronet_pairs(START, END, candidate='rf_rk',   blind_only=True)
print(f'AERONET-blind pairs — RF: {len(pairs_rf)}, kriging: {len(pairs_kr)}, rf_rk: {len(pairs_rk)}')

## Side-by-side metric panels

`compare_candidates` returns each candidate's full panel plus a `SUMMARY` row with the mean RMSE across `(site, season)` cells.

In [ ]:
vb.compare_candidates({'rf': pairs_rf, 'kriging': pairs_kr, 'rf_rk': pairs_rk})

## Paired skill — identical `(slot, site)` keys

Reports paired RMSE difference (`rmse_a − rmse_b`) and the paired t-test on absolute errors.  `p_value < 0.05` means the candidates differ significantly on the shared key set.

The two comparisons that matter for B3:
- **`rf_rk` vs `kriging`** — does regression kriging beat the kriging baseline? (negative `rmse_diff` = rf_rk better)
- **`rf_rk` vs `rf`** — how much does the kriged residual add over the bare RF drift?

In [ ]:
print('rf_rk vs kriging  (rmse_diff < 0 ⇒ rf_rk better):')
display(vb.paired_skill(pairs_rk, pairs_kr))
print('rf_rk vs rf       (rmse_diff < 0 ⇒ rf_rk better):')
display(vb.paired_skill(pairs_rk, pairs_rf))
print('rf vs kriging     (original baseline comparison):')
display(vb.paired_skill(pairs_rf, pairs_kr))

## Scatter — RF vs kriging vs rf_rk vs AERONET

In [ ]:
panels = [('RF gap-fill', pairs_rf), ('Kriging baseline', pairs_kr),
          ('RF + regression kriging', pairs_rk)]
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (label, df) in zip(axes, panels):
    if df.empty:
        ax.set_title(f'{label}: no matched pairs'); ax.set_visible(False); continue
    for site, marker in zip(df['site'].unique(), ('o', 's', '^', 'D')):
        sub = df[df['site'] == site]
        ax.scatter(sub['aer_aod'], sub['sat_aod'], alpha=0.6, label=site, marker=marker)
    lim = max(df[['aer_aod', 'sat_aod']].max().max(), 1.0)
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8)
    ax.set_xlabel('AERONET AOD'); ax.set_ylabel('Filled AOD'); ax.set_title(label)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()